In [ ]:
# Dependency bootstrap (auto-install missing packages)
import importlib.util
import subprocess
import sys

PKG_MAP = {
    'numpy': 'numpy',
    'matplotlib': 'matplotlib',
    'ipywidgets': 'ipywidgets',
}

missing = [pkg for mod, pkg in PKG_MAP.items() if importlib.util.find_spec(mod) is None]
if missing:
    print(f'Installing missing dependencies: {missing}')
    if sys.platform == 'emscripten':
        import micropip
        await micropip.install(missing)
    else:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])
print('Dependencies ready.')


# Logistic Regression Models and NLL Landscape

This notebook generates a 1D classification dataset, shows multiple logistic models, and visualizes the negative log-likelihood surface over `(w, b)`.

## Parameter Guide
- **Weight `w`**: Controls the slope of the logistic curve. Larger `|w|` makes the transition steeper.
- **Bias `b`**: Shifts the logistic curve left/right (intercept term).
- **Noise scale**: Standard deviation of Gaussian noise used when generating labels. Higher values make the data noisier and usually harder to fit.
- **Decision threshold**: Cutoff used to convert probabilities into class labels (`p >= threshold` => class 1).

### Buttons
- **Use minimum-NLL model**: Jumps to the model parameters that minimize NLL on the current generated dataset.
- **Regenerate data**: Draws a new noisy dataset using the current noise/threshold settings.
- **Reset demo**: Restores default slider values and default data configuration.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

In [ ]:
np.random.seed(0)
x_values = np.linspace(-5, 5, 100)
noise_scale = 1.5
threshold = 1.0
y_values = (x_values + np.random.normal(scale=noise_scale, size=x_values.shape) > threshold).astype(int)

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def probability(x, w, b):
    return sigmoid(w * x + b)

In [ ]:
w_values = np.linspace(-3, 3, 100)
b_values = np.linspace(-3, 3, 100)
W, B = np.meshgrid(w_values, b_values)

NLL = np.zeros(W.shape)
for i in range(W.shape[0]):
    for j in range(W.shape[1]):
        z = W[i, j] * x_values + B[i, j]
        probs = np.clip(sigmoid(z), 1e-6, 1 - 1e-6)
        NLL[i, j] = -np.mean(y_values * np.log(probs) + (1 - y_values) * np.log(1 - probs))

min_idx = np.unravel_index(np.argmin(NLL, axis=None), NLL.shape)
optimal_w = W[min_idx]
optimal_b = B[min_idx]
optimal_w, optimal_b, np.min(NLL)

In [ ]:
fig = plt.figure(figsize=(14, 6))

ax1 = fig.add_subplot(1, 2, 1)
ax1.scatter(x_values, y_values, color='blue', label='Data Points')
for w in np.linspace(-2, 2, 5):
    for b in np.linspace(-2, 2, 5):
        ax1.plot(x_values, probability(x_values, w, b), linestyle=':', color='gray', alpha=0.7)
ax1.plot(x_values, probability(x_values, optimal_w, optimal_b), color='red', linewidth=2,
         label=f'Optimal Model (w={optimal_w:.2f}, b={optimal_b:.2f})')
ax1.set_title('Logistic Regression Models')
ax1.set_xlabel('x')
ax1.set_ylabel('P(y=1|x)')
ax1.grid(alpha=0.25)
ax1.legend()

ax2 = fig.add_subplot(1, 2, 2, projection='3d')
ax2.plot_surface(W, B, NLL, cmap='viridis', alpha=0.85)
ax2.scatter(optimal_w, optimal_b, np.min(NLL), color='red', s=100,
            label=f'Minimum NLL (w={optimal_w:.2f}, b={optimal_b:.2f})')
ax2.set_title('NLL Loss Landscape')
ax2.set_xlabel('w')
ax2.set_ylabel('b')
ax2.set_zlabel('NLL')
ax2.legend()

plt.tight_layout()
plt.show()